# Singapore IT Jobs Data Cleaning

This notebook creates the shared clean dataset for the team project. The workflow was first tested on the first 50,000 rows. The final run shown in this notebook uses all 1,048,585 rows from `SGJobData.csv`. All final counts and exported files are based on the complete dataset.

The cleaning logic follows the course workflow: **find the issue, decide how to handle it, apply the rule, and verify the result**.


In [31]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================
# Import the libraries used to load, clean and transform the data.

import pandas as pd
import numpy as np
import json


In [32]:
# ============================================================
# STEP 1: LOAD THE DATA
# ============================================================
# Set USE_SAMPLE to True only when developing or testing the workflow.
# This final team run uses the complete CSV, so USE_SAMPLE is False.

USE_SAMPLE = False

if USE_SAMPLE:
    df = pd.read_csv("SGJobData.csv", nrows=50_000)
    print("Development sample loaded.")
else:
    df = pd.read_csv("SGJobData.csv", low_memory=False)
    print("Complete dataset loaded.")

print("Dataset shape:", df.shape)


Complete dataset loaded.
Dataset shape: (1048585, 22)


In [33]:
# Keep an unchanged copy for before-and-after comparisons.
raw = df.copy()

# Create the working DataFrame used for cleaning.
clean = df.copy()

print("Raw shape:", raw.shape)
print("Working shape:", clean.shape)

Raw shape: (1048585, 22)
Working shape: (1048585, 22)


In [34]:
# ============================================================
# STEP 2: INSPECT DATA QUALITY
# ============================================================

# Display the first five records.
display(clean.head())

# Display the number of rows and columns.
print("Dataset shape:", clean.shape)

# Display data types and non-null values.
clean.info()

# Calculate descriptive statistics.
display(
    clean.describe(
        include="all"
    ).T
)

,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary
0,"[{""id"":13,""category"":""Environment / Health""},{...",Permanent,2023-05-08,False,MCF-2023-0252866,2023-04-08,2023-03-30,2,5,151,...,NaN,Executive,WORKSTONE PTE. LTD.,2800,2000,Monthly,0,Closed,Food Technologist - Clementi | Entry Level | U...,2400.0
1,"[{""id"":21,""category"":""Information Technology""}]",Permanent,2023-05-08,False,MCF-2023-0273977,2023-04-08,2023-04-08,0,0,55,...,NaN,Executive,TRUST RECRUIT PTE. LTD.,5500,4000,Monthly,0,Closed,"Software Engineer (Fab Support) (Java, CIM, Up...",4750.0
2,"[{""id"":33,""category"":""Repair and Maintenance""}]",Full Time,2023-04-22,False,MCF-2023-0273994,2023-04-08,2023-04-08,0,7,99,...,NaN,Senior Executive,PU TIEN SERVICES PTE. LTD.,4600,3800,Monthly,0,Closed,Senior Technician,4200.0
3,"[{""id"":21,""category"":""Information Technology""}]",Permanent,2023-05-08,False,MCF-2023-0273991,2023-04-08,2023-04-08,0,6,113,...,NaN,Senior Executive,TRUST RECRUIT PTE. LTD.,10000,5000,Monthly,0,Closed,"Senior .NET Developer (.NET Core, MVC, MVVC, S...",7500.0
4,"[{""id"":2,""category"":""Admin / Secretarial""}]",Full Time,2023-05-08,False,MCF-2023-0273976,2023-04-08,2023-04-08,0,3,99,...,NaN,Non-executive,EATZ CATERING SERVICES PTE. LTD.,3400,2400,Monthly,0,Closed,Sales / Admin Cordinator,2900.0


Dataset shape: (1048585, 22)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048585 entries, 0 to 1048584
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  object 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1048585 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1048585 non-null  int64  
 8   metadata_totalNumberJobApplication  1048585 non-null  int64  
 9   metadata_totalNumberOfView          1048585 non-null  int64  
 10  minimumYearsExperience              1048585 non-n

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
categories,1044597,21125,"[{""id"":21,""category"":""Information Technology""}]",92869,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employmentTypes,1044597,8,Permanent,458139,NaN,NaN,NaN,NaN,NaN,NaN,NaN
metadata_expiryDate,1044597,453,2023-07-28,4487,NaN,NaN,NaN,NaN,NaN,NaN,NaN
metadata_isPostedOnBehalf,1048585,2,False,986717,NaN,NaN,NaN,NaN,NaN,NaN,NaN
metadata_jobPostId,1044597,1044597,MCF-2023-0252866,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
metadata_newPostingDate,1044597,431,2023-06-09,4508,NaN,NaN,NaN,NaN,NaN,NaN,NaN
metadata_originalPostingDate,1044597,603,2023-07-14,4029,NaN,NaN,NaN,NaN,NaN,NaN,NaN
metadata_repostCount,1048585.0,NaN,NaN,NaN,0.054723,0.282268,0.0,0.0,0.0,0.0,2.0
metadata_totalNumberJobApplication,1048585.0,NaN,NaN,NaN,2.136571,10.626123,0.0,0.0,0.0,1.0,1342.0
metadata_totalNumberOfView,1048585.0,NaN,NaN,NaN,26.74536,82.620014,0.0,1.0,4.0,17.0,8190.0


In [35]:
# Count and calculate the percentage of missing values.

missing_summary = pd.DataFrame({
    "missing_count": clean.isna().sum(),
    "missing_percentage": (
        clean.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_summary)

,missing_count,missing_percentage
occupationId,1048585,100.00
categories,3988,0.38
metadata_expiryDate,3988,0.38
title,3988,0.38
metadata_jobPostId,3988,0.38
metadata_newPostingDate,3988,0.38
metadata_originalPostingDate,3988,0.38
status_jobStatus,3988,0.38
salary_type,3988,0.38
employmentTypes,3988,0.38


In [36]:
# Check complete duplicate rows and duplicate Job IDs.

print(
    "Complete duplicate rows:",
    clean.duplicated().sum()
)

print(
    "Duplicate non-missing Job IDs:",
    clean["metadata_jobPostId"]
         .dropna()
         .duplicated()
         .sum()
)

Complete duplicate rows: 3987
Duplicate non-missing Job IDs: 0


In [37]:
# ============================================================
# STEP 3: REMOVE STRUCTURALLY INVALID DATA
# ============================================================

# Identify rows that do not represent valid job postings.

structurally_empty = (
    clean["title"].isna()
    & clean["categories"].isna()
    & clean["metadata_originalPostingDate"].isna()
    & clean["salary_minimum"].fillna(0).eq(0)
    & clean["salary_maximum"].fillna(0).eq(0)
    & clean["numberOfVacancies"].fillna(0).eq(0)
)

print(
    "Structurally empty rows:",
    structurally_empty.sum()
)

display(
    clean.loc[
        structurally_empty
    ].head()
)

Structurally empty rows: 3988


,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary
197478,NaN,NaN,NaN,False,NaN,NaN,NaN,0,0,0,...,NaN,NaN,NaN,0,0,NaN,0,NaN,NaN,0.0
197480,NaN,NaN,NaN,False,NaN,NaN,NaN,0,0,0,...,NaN,NaN,NaN,0,0,NaN,0,NaN,NaN,0.0
197485,NaN,NaN,NaN,False,NaN,NaN,NaN,0,0,0,...,NaN,NaN,NaN,0,0,NaN,0,NaN,NaN,0.0
197488,NaN,NaN,NaN,False,NaN,NaN,NaN,0,0,0,...,NaN,NaN,NaN,0,0,NaN,0,NaN,NaN,0.0
197502,NaN,NaN,NaN,False,NaN,NaN,NaN,0,0,0,...,NaN,NaN,NaN,0,0,NaN,0,NaN,NaN,0.0


In [38]:
# Remove structurally empty records.

rows_before = len(clean)

clean = clean.loc[
    ~structurally_empty
].copy()

print("Rows before:", rows_before)
print("Rows after:", len(clean))
print("Rows removed:", rows_before - len(clean))

Rows before: 1048585
Rows after: 1044597
Rows removed: 3988


In [39]:
# Remove occupationId only when every value is missing.

print(
    "Missing occupationId:",
    clean["occupationId"].isna().sum()
)

if clean["occupationId"].isna().all():
    clean = clean.drop(
        columns=["occupationId"]
    )

    print("occupationId removed.")

else:
    print("occupationId retained.")

Missing occupationId: 1044597
occupationId removed.


In [40]:
# ============================================================
# STEP 4: CONVERT DATES AND PREPARE TEXT
# ============================================================

# Convert text columns into proper datetime columns.

date_columns = [
    "metadata_expiryDate",
    "metadata_newPostingDate",
    "metadata_originalPostingDate"
]

for column in date_columns:
    clean[column] = pd.to_datetime(
        clean[column],
        errors="coerce"
    )

# Create a posting-month column for Power BI.
clean["posting_month"] = (
    clean["metadata_originalPostingDate"]
    .dt.to_period("M")
    .astype(str)
)

print(
    "Earliest posting date:",
    clean["metadata_originalPostingDate"].min()
)

print(
    "Latest posting date:",
    clean["metadata_originalPostingDate"].max()
)

Earliest posting date: 2022-10-03 00:00:00
Latest posting date: 2024-05-29 00:00:00


In [41]:
# Preserve the original title and create a cleaned version.

clean["title_raw"] = clean["title"]

clean["title_clean"] = (
    clean["title"]
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

In [42]:
# Preserve the original position level and remove extra spaces.

clean["position_level_raw"] = (
    clean["positionLevels"]
)

clean["position_level"] = (
    clean["positionLevels"]
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

display(
    clean["position_level"]
    .value_counts(dropna=False)
)

Executive            253701
Junior Executive     167656
Non-executive        131608
Fresh/entry level    118661
Professional         112208
Manager              110122
Senior Executive     100459
Middle Management     27375
Senior Management     22807
Name: position_level, dtype: int64

In [43]:
# ============================================================
# STEP 5: PARSE CATEGORIES AND FILTER IT JOBS
# ============================================================

# Convert the categories JSON text into a Python list.

def parse_categories(value):

    if pd.isna(value):
        return []

    try:
        result = json.loads(value)

        if isinstance(result, list):
            return result

        return []

    except (json.JSONDecodeError, TypeError):
        return []

In [44]:
# Parse the categories field.

clean["category_items"] = (
    clean["categories"]
    .apply(parse_categories)
)

# Extract category names.

clean["category_list"] = (
    clean["category_items"]
    .apply(
        lambda items: [
            item.get("category")
            for item in items
            if isinstance(item, dict)
            and item.get("category") is not None
        ]
    )
)

# Count categories per job.

clean["category_count"] = (
    clean["category_list"]
    .apply(len)
)

display(
    clean[
        [
            "metadata_jobPostId",
            "categories",
            "category_list"
        ]
    ].head()
)

,metadata_jobPostId,categories,category_list
0,MCF-2023-0252866,"[{""id"":13,""category"":""Environment / Health""},{...","[Environment / Health, Manufacturing, Sciences..."
1,MCF-2023-0273977,"[{""id"":21,""category"":""Information Technology""}]",[Information Technology]
2,MCF-2023-0273994,"[{""id"":33,""category"":""Repair and Maintenance""}]",[Repair and Maintenance]
3,MCF-2023-0273991,"[{""id"":21,""category"":""Information Technology""}]",[Information Technology]
4,MCF-2023-0273976,"[{""id"":2,""category"":""Admin / Secretarial""}]",[Admin / Secretarial]


In [45]:
# Keep jobs classified under Information Technology.

is_it_job = (
    clean["category_list"]
    .apply(
        lambda categories:
            "Information Technology"
            in categories
    )
)

print("Rows before IT filter:", len(clean))
print("IT job rows:", is_it_job.sum())

clean = clean.loc[
    is_it_job
].copy()

print("Rows after IT filter:", len(clean))

Rows before IT filter: 1044597
IT job rows: 140866
Rows after IT filter: 140866


In [46]:
# ============================================================
# STEP 6: CLEAN SALARY DATA
# ============================================================

# Check the salary types before filtering or removing the column.

display(
    clean["salary_type"]
    .value_counts(dropna=False)
)

Monthly    140866
Name: salary_type, dtype: int64

In [47]:
# Keep only Monthly salary records.

clean = clean.loc[
    clean["salary_type"].eq("Monthly")
].copy()

In [48]:
# Convert salary columns to numeric data.

salary_columns = [
    "salary_minimum",
    "salary_maximum",
    "average_salary"
]

for column in salary_columns:
    clean[column] = pd.to_numeric(
        clean[column],
        errors="coerce"
    )

# Preserve original salary values.

clean["salary_minimum_raw"] = (
    clean["salary_minimum"]
)

clean["salary_maximum_raw"] = (
    clean["salary_maximum"]
)

clean["average_salary_raw"] = (
    clean["average_salary"]
)

In [49]:
# Remove records with maximum salary below 800.

rows_before = len(clean)

clean = clean.loc[
    clean["salary_maximum"] >= 800
].copy()

print(
    "Maximum salary below 800 removed:",
    rows_before - len(clean)
)

Maximum salary below 800 removed: 192


In [50]:
# Remove remaining records with minimum salary at or below 700.

rows_before = len(clean)

clean = clean.loc[
    clean["salary_minimum"] > 700
].copy()

print(
    "Minimum salary at or below 700 removed:",
    rows_before - len(clean)
)

Minimum salary at or below 700 removed: 167


In [51]:
# Identify possible annual salaries entered into a monthly field.

clean["salary_ratio_before_correction"] = (
    clean["salary_maximum"]
    / clean["salary_minimum"]
)

annual_error = (
    clean["salary_ratio_before_correction"]
    >= 12
)

salary_review = clean.loc[
    annual_error,
    [
        "metadata_jobPostId",
        "title_clean",
        "position_level",
        "salary_minimum",
        "salary_maximum",
        "salary_ratio_before_correction"
    ]
].copy()

salary_review["corrected_maximum"] = (
    salary_review["salary_maximum"]
    / 12
)

salary_review["corrected_ratio"] = (
    salary_review["corrected_maximum"]
    / salary_review["salary_minimum"]
)

salary_review["correction_accepted"] = (
    salary_review["corrected_ratio"]
    .between(1, 3)
)

display(salary_review)

,metadata_jobPostId,title_clean,position_level,salary_minimum,salary_maximum,salary_ratio_before_correction,corrected_maximum,corrected_ratio,correction_accepted
9932,MCF-2023-0279935,Senior Software Engineer [Java/ Oracle/ Up to ...,Senior Executive,8000,110000,13.750000,9166.666667,1.145833,True
50912,MCF-2023-0398894,Business Development Director,Senior Management,9000,120000,13.333333,10000.000000,1.111111,True
85599,MCF-2023-0362920,Staff Software Engineer (Java),Professional,10000,120000,12.000000,10000.000000,1.000000,True
107399,MCF-2023-0407294,Sales Agent,Fresh/entry level,1000,15000,15.000000,1250.000000,1.250000,True
141561,MCF-2023-0440158,Client Solutions - Manager/Senior Manager/Asso...,Manager,6500,130000,20.000000,10833.333333,1.666667,True
150606,MCF-2023-0499912,BI and Data Modeler,Middle Management,10000,130000,13.000000,10833.333333,1.083333,True
153739,MCF-2023-0492915,Backend Developer,Professional,6000,72000,12.000000,6000.000000,1.000000,True
184281,MCF-2023-0461172,Operational Risk – Compliance & Risk Managemen...,Professional,1000,16000,16.000000,1333.333333,1.333333,True
189778,MCF-2023-0454021,VP - Lead IT PMO,Manager,1000,13000,13.000000,1083.333333,1.083333,True
213211,MCF-2023-0515108,PMO Software Application,Professional,7000,90000,12.857143,7500.000000,1.071429,True


In [52]:
# Correct records that produce a plausible monthly salary range.

accepted_ids = salary_review.loc[
    salary_review["correction_accepted"],
    "metadata_jobPostId"
]

accepted_correction = (
    clean["metadata_jobPostId"]
    .isin(accepted_ids)
)

clean["salary_maximum"] = (
    clean["salary_maximum"]
    .astype(float)
)

clean.loc[
    accepted_correction,
    "salary_maximum"
] = (
    clean.loc[
        accepted_correction,
        "salary_maximum"
    ] / 12
)

clean["salary_maximum_corrected"] = (
    accepted_correction
)

In [53]:
# Remove salary records that remain implausible after testing.

rejected_ids = salary_review.loc[
    ~salary_review["correction_accepted"],
    "metadata_jobPostId"
]

clean = clean.loc[
    ~clean["metadata_jobPostId"]
    .isin(rejected_ids)
].copy()

In [54]:
# Calculate final salary measures after cleaning minimum and maximum salary.

clean["salary_midpoint"] = (
    clean["salary_minimum"]
    + clean["salary_maximum"]
) / 2

clean["salary_band_width"] = (
    clean["salary_maximum"]
    - clean["salary_minimum"]
)

# Update average_salary so it agrees with the cleaned salary boundaries.
# The original value remains available in average_salary_raw.
clean["average_salary"] = clean["salary_midpoint"]


In [55]:
# ============================================================
# STEP 7: FLAG ISSUES, VALIDATE AND EXPORT
# ============================================================

# Flag experience values for review without changing the source value.

clean["experience_review_flag"] = (
    clean["minimumYearsExperience"]
    >= 30
)

# Keep all dates but identify records from May 2023 onwards.

clean["from_may_2023_flag"] = (
    clean["metadata_originalPostingDate"]
    >= "2023-05-01"
)

In [56]:
# Create one row per job-category relationship.

job_categories = (
    clean[
        [
            "metadata_jobPostId",
            "category_list"
        ]
    ]
    .explode("category_list")
    .rename(
        columns={
            "category_list": "category"
        }
    )
    .dropna(subset=["category"])
    .drop_duplicates()
    .reset_index(drop=True)
)

In [57]:
# Run final validation checks.

assert clean["metadata_jobPostId"].notna().all()
assert clean["metadata_jobPostId"].is_unique
assert (clean["salary_minimum"] > 0).all()
assert (clean["salary_maximum"] > 0).all()
assert (clean["salary_minimum"] <= clean["salary_maximum"]).all()
assert clean["salary_midpoint"].notna().all()
assert clean["average_salary"].equals(clean["salary_midpoint"])
assert clean["salary_type"].eq("Monthly").all()

print("All validation checks passed.")


All validation checks passed.


In [58]:
# ============================================================
# EXPORT THE FINAL TEAM DATASETS
# ============================================================
# File names change automatically between sample and full-data runs.

if USE_SAMPLE:
    jobs_filename = "jobs_clean_sample.csv"
    categories_filename = "job_categories_sample.csv"
    corrections_filename = "salary_corrections_sample.csv"
else:
    jobs_filename = "jobs_clean.csv"
    categories_filename = "job_categories.csv"
    corrections_filename = "salary_corrections.csv"

# Remove the temporary parsed-object column before CSV export.
clean = clean.drop(columns=["category_items"])

clean.to_csv(jobs_filename, index=False)
job_categories.to_csv(categories_filename, index=False)
salary_review.to_csv(corrections_filename, index=False)

print("Original rows:", len(raw))
print("Structurally empty rows removed:", structurally_empty.sum())
print("Final clean IT rows:", len(clean))
print("Unique final Job IDs:", clean["metadata_jobPostId"].nunique())
print("Salary corrections accepted:", salary_review["correction_accepted"].sum())
print("Salary candidates removed:", (~salary_review["correction_accepted"]).sum())
print("Saved:", jobs_filename)
print("Saved:", categories_filename)
print("Saved:", corrections_filename)


Original rows: 1048585
Structurally empty rows removed: 3988
Final clean IT rows: 140506
Unique final Job IDs: 140506
Salary corrections accepted: 54
Salary candidates removed: 1
Saved: jobs_clean.csv
Saved: job_categories.csv
Saved: salary_corrections.csv


## Final team output

This notebook completed the full-data cleaning workflow and produced:

- `jobs_clean.csv`: one row per clean IT job posting
- `job_categories.csv`: one row per job-category relationship
- `salary_corrections.csv`: the audit table for suspected annual/monthly salary errors

The clean master dataset retains all posting dates. The `from_may_2023_flag` field allows time-trend analyses to select May 2023 onward without deleting earlier records. Experience values of 30 years or more are flagged for review rather than overwritten.
